# Amatrice paper experiment guide (LSTM / coherence-source comparison)

This notebook is a dedicated experiment guide for the workflow you described:

1. Crop the Amatrice interferogram pairs to `-l 42.6 42.7 -L 13.2 13.4`.
2. Generate auxiliary INT-derived products:
   - `unfilt_fine.cor`
   - `underamp_unfilt_fine.cor`
   - `underamp_unfilt_fine_circ.cor`
   - `filt_fine.std`
3. Build separate datasets for:
   - `fine.cor.full` (band 2 coherence)
   - `filt_fine.cor`
   - `unfilt_fine.cor`
   - `underamp_unfilt_fine.cor`
   - `underamp_unfilt_fine_circ.cor`
   - `filt_fine.std`
4. Train LSTM models with and without timestamp features.
5. Compute **non-zscore** NDI scores.

> Important:
> - Coherence score uses `(pred - obs) / (pred + obs + eps)`.
> - Phase-STD score uses `(obs - pred) / (pred + obs + eps)`.
> - Therefore `filt_fine.std` must be trained/scored with `--timeseries-metric phase_std`.

> Troubleshooting: if you still see `NameError: epoch_loss is not defined`, you are running an outdated local copy of the repository rather than the fixed version.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess

BASE_DIR = Path('/data6/WORKDIR/AmatriceSenDT22/merged/interferograms')
GEOM_REF_DIR = Path('/data6/WORKDIR/AmatriceSenDT22/merged/geom_reference')
CROPPED_DIR = BASE_DIR / 'cropped_paper_bbox'
EVENT_DATE = '20160824'
NEXT_DATE = '20160821_20160914'
LAT_MIN, LAT_MAX = 42.6, 42.7
LON_MIN, LON_MAX = 13.2, 13.4
PARAM_FILE = CROPPED_DIR / 'amatrice_lstm_params.json'


def run_cmd(cmd: str, env=None):
    print(f"[RUN] {cmd}")
    merged_env = os.environ.copy()
    merged_env['PYTHONUNBUFFERED'] = '1'
    if env:
        merged_env.update(env)
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True, env=merged_env)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed with code {result.returncode}')


try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
    DEVICE_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else 'CPU'
except Exception:
    GPU_AVAILABLE = False
    DEVICE_NAME = 'CPU'

print('BASE_DIR =', BASE_DIR)
print('CROPPED_DIR =', CROPPED_DIR)
print('Training device policy = CUDA if available else CPU')
print('Detected device =', DEVICE_NAME)


## 1. Crop the requested pairs to the paper bbox


In [ ]:
run_cmd(
    f"python -m insar_pipeline.app --step crop "
    f"--base-dir {BASE_DIR} "
    f"--geom-reference-dir {GEOM_REF_DIR} "
    f"--cropped-dir {CROPPED_DIR} "
    f"--output-dir {CROPPED_DIR} "
    f"--lat-min {LAT_MIN} --lat-max {LAT_MAX} "
    f"--lon-min {LON_MIN} --lon-max {LON_MAX}"
)


## 2. Generate INT-derived auxiliary products


In [ ]:
run_cmd(
    f"python -m insar_pipeline.app --step prepare_int_aux "
    f"--cropped-dir {CROPPED_DIR} "
    f"--output-dir {CROPPED_DIR} "
    f"--aux-corr-win 5 --aux-phsig-win 5 "
    f"--aux-variance-win 5 --aux-variance-looks 3.0 "
    f"--aux-block-lines 512"
)


## 3. Inspect prepared products


In [ ]:
run_cmd(f"find {CROPPED_DIR} -maxdepth 1 -type f | sort")


## 4. Build all requested datasets


In [ ]:
DATASET_SPECS = [
    ('fine.cor.full', 'coherence', 'dataset_rnn_fine_cor_full'),
    ('filt_fine.cor', 'coherence', 'dataset_rnn_filt_fine_cor'),
    ('unfilt_fine.cor', 'coherence', 'dataset_rnn_unfilt_fine_cor'),
    ('underamp_unfilt_fine.cor', 'coherence', 'dataset_rnn_underamp_unfilt_fine_cor'),
    ('underamp_unfilt_fine_circ.cor', 'coherence', 'dataset_rnn_underamp_unfilt_fine_circ_cor'),
    ('filt_fine.std', 'phase_std', 'dataset_rnn_filt_fine_std'),
]

for observation_file, metric, dataset_name in DATASET_SPECS:
    extra = "--looks 3.0" if observation_file == 'filt_fine.std' else ""
    run_cmd(
        f"python -m insar_pipeline.app --step build_dataset "
        f"--cropped-dir {CROPPED_DIR} "
        f"--output-dir {CROPPED_DIR} "
        f"--event-date {EVENT_DATE} "
        f"--input-source cor "
        f"--observation-file {observation_file} "
        f"--dataset-name {dataset_name} {extra}"
    )


## 5. LSTM training: shared hyperparameters + timestamp experiment design

- All datasets below use the **same LSTM/training hyperparameters** so the comparison is fair.
- `filt_fine.std` runs **both** `time` and `notime`.
- All coherence-based datasets run **only** `notime` as requested.
- The CLI/model code will automatically select **GPU when CUDA is available**, otherwise it falls back to CPU.
- Training progress is printed epoch by epoch in the notebook output.


In [ ]:
COMMON_TRAINING_CONFIG = {
    'global': {},
    'rnn': {
        'epochs': 15,
        'train_batch_size': 128,
        'pred_batch_size': 256,
        'lr': 1e-3,
        'ts_model': 'lstm',
        'optimizer': 'adam',
        'weight_decay': 0.0,
        'max_grad_norm': 1.0,
        'rnn_hidden_dim': 64,
        'rnn_num_layers': 2,
        'rnn_dropout': 0.1,
    },
}

CROPPED_DIR.mkdir(parents=True, exist_ok=True)
PARAM_FILE.write_text(json.dumps(COMMON_TRAINING_CONFIG, indent=2), encoding='utf-8')
print('Saved shared training config to', PARAM_FILE)
print(json.dumps(COMMON_TRAINING_CONFIG, indent=2))

EXPERIMENTS = [
    {
        'observation_file': 'filt_fine.std',
        'metric': 'phase_std',
        'dataset_name': 'dataset_rnn_filt_fine_std',
        'timestamp_modes': ['time', 'notime'],
    },
    {
        'observation_file': 'fine.cor.full',
        'metric': 'coherence',
        'dataset_name': 'dataset_rnn_fine_cor_full',
        'timestamp_modes': ['notime'],
    },
    {
        'observation_file': 'filt_fine.cor',
        'metric': 'coherence',
        'dataset_name': 'dataset_rnn_filt_fine_cor',
        'timestamp_modes': ['notime'],
    },
    {
        'observation_file': 'unfilt_fine.cor',
        'metric': 'coherence',
        'dataset_name': 'dataset_rnn_unfilt_fine_cor',
        'timestamp_modes': ['notime'],
    },
    {
        'observation_file': 'underamp_unfilt_fine.cor',
        'metric': 'coherence',
        'dataset_name': 'dataset_rnn_underamp_unfilt_fine_cor',
        'timestamp_modes': ['notime'],
    },
    {
        'observation_file': 'underamp_unfilt_fine_circ.cor',
        'metric': 'coherence',
        'dataset_name': 'dataset_rnn_underamp_unfilt_fine_circ_cor',
        'timestamp_modes': ['notime'],
    },
]

print('Experiment matrix:')
for exp in EXPERIMENTS:
    print(
        f"- {exp['observation_file']:30s} metric={exp['metric']:10s} "
        f"dataset={exp['dataset_name']} timestamp_modes={exp['timestamp_modes']}"
    )

for exp in EXPERIMENTS:
    for timestamp_flag in exp['timestamp_modes']:
        disable = '--disable-timestamp' if timestamp_flag == 'notime' else ''
        artifact_tag = exp['observation_file'].replace('.', '_')
        print('\n' + '=' * 80)
        print('Training run')
        print('  observation_file =', exp['observation_file'])
        print('  metric           =', exp['metric'])
        print('  dataset_dir      =', CROPPED_DIR / exp['dataset_name'])
        print('  timestamp_mode   =', timestamp_flag)
        print('  param_file       =', PARAM_FILE)
        print('  device_policy    = cuda-if-available-else-cpu')
        print('=' * 80)
        run_cmd(
            f"python -m insar_pipeline.app --step train_predict "
            f"--dataset-dir {shlex.quote(str(CROPPED_DIR / exp['dataset_name']))} "
            f"--output-dir {shlex.quote(str(CROPPED_DIR))} "
            f"--next-date {NEXT_DATE} "
            f"--timeseries-metric {exp['metric']} "
            f"--ts-model lstm "
            f"--artifact-tag {artifact_tag} "
            f"--param-file {shlex.quote(str(PARAM_FILE))} {disable}"
        )


## 6. Compute non-zscore NDI scores with the same experiment matrix

- `phase_std` score uses the phase-STD direction handled by the repository scoring logic.
- `coherence` score uses the coherence-oriented normalized difference index.
- The score loop follows the same timestamp matrix as the training loop above.


In [ ]:
for exp in EXPERIMENTS:
    for timestamp_flag in exp['timestamp_modes']:
        artifact_tag = exp['observation_file'].replace('.', '_')
        disable = '--disable-timestamp' if timestamp_flag == 'notime' else ''
        print('\n' + '-' * 80)
        print('Score run')
        print('  observation_file =', exp['observation_file'])
        print('  metric           =', exp['metric'])
        print('  dataset_dir      =', CROPPED_DIR / exp['dataset_name'])
        print('  timestamp_mode   =', timestamp_flag)
        print('  score_mode       = ndi')
        print('-' * 80)
        run_cmd(
            f"python -m insar_pipeline.app --step score "
            f"--dataset-dir {shlex.quote(str(CROPPED_DIR / exp['dataset_name']))} "
            f"--predict-dir {shlex.quote(str(CROPPED_DIR / 'predict'))} "
            f"--output-dir {shlex.quote(str(CROPPED_DIR))} "
            f"--timeseries-metric {exp['metric']} "
            f"--score-mode ndi "
            f"--ts-model lstm "
            f"--artifact-tag {artifact_tag} {disable}"
        )


## 7. Optional geocoded outputs for selected scores


In [ ]:
run_cmd(
    f"python -m insar_pipeline.app --step output "
    f"--predict-dir {CROPPED_DIR / 'predict'} "
    f"--output-dir {CROPPED_DIR} "
    f"--lat-file {CROPPED_DIR / 'lat_cropped.rdr'} "
    f"--lon-file {CROPPED_DIR / 'lon_cropped.rdr'} "
    f"--subset-params '-l 42.6 42.7 -L 13.2 13.4'"
)


## 8. Suggested result table export


In [ ]:
run_cmd(f"find {CROPPED_DIR / 'predict'} -maxdepth 1 -type f | sort")
